# 🔥 Real-Time Fraud Detection - Pipeline Complet

## Architecture simulée:

```
┌──────────────┐    ┌──────────────┐    ┌──────────────┐    ┌──────────────┐    ┌──────────────┐
│   KAFKA      │───▶│    SPARK     │───▶│    HDFS      │───▶│   IMPALA     │───▶│ DASHBOARD    │
│  Producer    │    │  Streaming   │    │  (Parquet)   │    │   (SQL)      │    │ Visualisation│
└──────────────┘    └──────────────┘    └──────────────┘    └──────────────┘    └──────────────┘
```

### Ce notebook démontre:
1. **Kafka**: Simulation producer/consumer avec topics
2. **Spark Streaming**: Traitement micro-batch temps réel
3. **HDFS**: Stockage Parquet partitionné
4. **Impala/SQL**: Requêtes analytiques massives
5. **MLlib**: Modèle de détection de fraude
6. **Visualisation**: Dashboards interactifs
7. **Airflow**: Simulation DAG

---
# 🛠️ INSTALLATION

In [ ]:
!pip install pyspark==3.4.1 -q
!pip install plotly pandas pyarrow -q
print("✅ Installation terminée!")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# IMPORTS - ATTENTION: On importe PySpark avec alias pour éviter conflits
# ══════════════════════════════════════════════════════════════════════

import os
import json
import time
import random
import shutil
from datetime import datetime
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# PySpark - imports spécifiques (pas de *)
from pyspark.sql import SparkSession
from pyspark.sql import functions as F  # IMPORTANT: alias F pour éviter conflits
from pyspark.sql.types import *
from pyspark.sql.window import Window

# ML
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml import Pipeline

# Visualisation
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print("✅ Imports réussis!")

In [ ]:
# Créer SparkSession
spark = SparkSession.builder \
    .appName("RealTimeFraudDetection") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

sc = spark.sparkContext
sc.setLogLevel("ERROR")

print(f"✅ Spark {spark.version} initialisé")
print(f"   Master: {sc.master}")

In [ ]:
# Créer structure HDFS simulée
HDFS_BASE = "/content/hdfs_simulation"
HDFS_RAW = f"{HDFS_BASE}/raw/transactions"
HDFS_PROCESSED = f"{HDFS_BASE}/processed/transactions"
HDFS_ALERTS = f"{HDFS_BASE}/alerts"
HDFS_MODELS = f"{HDFS_BASE}/models"

if os.path.exists(HDFS_BASE):
    shutil.rmtree(HDFS_BASE)

for path in [HDFS_RAW, HDFS_PROCESSED, HDFS_ALERTS, HDFS_MODELS]:
    os.makedirs(path, exist_ok=True)

print(f"✅ Structure HDFS simulée:")
print(f"   📁 {HDFS_BASE}/")
print(f"   ├── raw/transactions")
print(f"   ├── processed/transactions")
print(f"   ├── alerts")
print(f"   └── models")

---
# 📨 PARTIE 1: KAFKA SIMULATION

Simulation Topics, Producer, Consumer

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# KAFKA SIMULATOR
# ══════════════════════════════════════════════════════════════════════

class KafkaSimulator:
    """Simule un broker Kafka avec topics, producers et consumers"""
    
    def __init__(self):
        self.topics = defaultdict(list)
        self.offsets = defaultdict(int)
        self.partitions = defaultdict(lambda: defaultdict(list))
        
    def create_topic(self, name, num_partitions=3):
        for i in range(num_partitions):
            self.partitions[name][i] = []
        print(f"✅ Topic '{name}' créé ({num_partitions} partitions)")
        
    def produce(self, topic, key, value):
        partition = hash(key) % len(self.partitions[topic])
        message = {
            "key": key,
            "value": value,
            "partition": partition,
            "offset": len(self.partitions[topic][partition]),
            "timestamp": int(time.time() * 1000)
        }
        self.partitions[topic][partition].append(message)
        self.topics[topic].append(message)
        return message
    
    def consume(self, topic, consumer_group, max_messages=100):
        offset_key = f"{topic}:{consumer_group}"
        current_offset = self.offsets[offset_key]
        messages = self.topics[topic][current_offset:current_offset + max_messages]
        self.offsets[offset_key] = current_offset + len(messages)
        return messages
    
    def get_stats(self, topic):
        return {
            "topic": topic,
            "total_messages": len(self.topics[topic]),
            "partitions": {p: len(msgs) for p, msgs in self.partitions[topic].items()}
        }

# Créer broker Kafka
kafka = KafkaSimulator()
kafka.create_topic("transactions", num_partitions=6)
kafka.create_topic("fraud-alerts", num_partitions=3)

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# TRANSACTION PRODUCER
# ══════════════════════════════════════════════════════════════════════

class TransactionProducer:
    """Producteur Kafka de transactions"""
    
    LOCATIONS = ["Paris", "London", "New York", "Tokyo", "Singapore", "Dubai"]
    CATEGORIES = ["RETAIL", "GROCERY", "RESTAURANT", "ONLINE", "TRAVEL",
                  "ENTERTAINMENT", "UTILITIES", "HEALTHCARE", "GAMBLING", "CRYPTO"]
    CHANNELS = ["ONLINE", "POS", "ATM", "MOBILE"]
    
    def __init__(self, kafka_broker):
        self.kafka = kafka_broker
        self.tx_count = 0
        
    def generate_normal_transaction(self):
        # Utiliser random.gauss directement (pas de conflit avec PySpark)
        amount = random.gauss(200, 150)
        amount = 5 if amount < 5 else amount  # min 5
        
        return {
            "transactionId": f"TX{int(time.time()*1000)}-{random.randint(1000,9999)}",
            "customerId": f"CUST{random.randint(1, 5000):04d}",
            "merchantId": f"MERCH{random.randint(1, 500):03d}",
            "amount": round(amount, 2),
            "currency": random.choice(["EUR", "USD", "GBP"]),
            "channel": random.choice(self.CHANNELS),
            "location": random.choice(self.LOCATIONS),
            "timestamp": int(time.time() * 1000),
            "cardType": random.choice(["CREDIT", "DEBIT"]),
            "isInternational": random.random() < 0.1,
            "merchantCategory": random.choice(self.CATEGORIES[:8]),
            "previousBalance": round(random.uniform(500, 15000), 2),
            "isFraud": False
        }
    
    def generate_fraudulent_transaction(self):
        tx = self.generate_normal_transaction()
        tx["isFraud"] = True
        
        fraud_type = random.choice(["high_amount", "risky_merchant", "international"])
        
        if fraud_type == "high_amount":
            tx["amount"] = round(random.uniform(5000, 15000), 2)
        elif fraud_type == "risky_merchant":
            tx["merchantCategory"] = random.choice(["GAMBLING", "CRYPTO"])
            tx["amount"] = round(random.uniform(1000, 8000), 2)
        else:
            tx["isInternational"] = True
            tx["amount"] = round(random.uniform(1000, 5000), 2)
            
        return tx
    
    def produce(self, n_transactions, fraud_rate=0.03):
        produced = []
        fraud_count = 0
        
        for i in range(n_transactions):
            if random.random() < fraud_rate:
                tx = self.generate_fraudulent_transaction()
                fraud_count += 1
            else:
                tx = self.generate_normal_transaction()
            
            self.kafka.produce("transactions", tx["transactionId"], json.dumps(tx))
            produced.append(tx)
            self.tx_count += 1
            
        print(f"📤 {n_transactions} transactions ({fraud_count} fraudes = {fraud_count/n_transactions*100:.1f}%)")
        return produced

producer = TransactionProducer(kafka)

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# PRODUIRE DES TRANSACTIONS
# ══════════════════════════════════════════════════════════════════════

print("="*60)
print("   KAFKA PRODUCER - Génération de transactions")
print("="*60)

all_transactions = []
for batch in range(5):
    print(f"\n📦 Batch {batch+1}/5:")
    txs = producer.produce(2000, fraud_rate=0.03)
    all_transactions.extend(txs)

print(f"\n{'='*60}")
print(f"📊 Total: {len(all_transactions)} transactions")

stats = kafka.get_stats("transactions")
print(f"\n📈 Topic 'transactions': {stats['total_messages']} messages")
print(f"   Partitions: {stats['partitions']}")

---
# ⚡ PARTIE 2: SPARK STREAMING

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# SPARK STREAMING PROCESSOR
# ══════════════════════════════════════════════════════════════════════

class SparkStreamingProcessor:
    """Simule Spark Structured Streaming"""
    
    def __init__(self, spark_session, kafka_broker):
        self.spark = spark_session
        self.kafka = kafka_broker
        self.processed_count = 0
        self.alerts_count = 0
        
        self.schema = StructType([
            StructField("transactionId", StringType(), False),
            StructField("customerId", StringType(), False),
            StructField("merchantId", StringType(), False),
            StructField("amount", DoubleType(), False),
            StructField("currency", StringType(), False),
            StructField("channel", StringType(), False),
            StructField("location", StringType(), False),
            StructField("timestamp", LongType(), False),
            StructField("cardType", StringType(), False),
            StructField("isInternational", BooleanType(), False),
            StructField("merchantCategory", StringType(), False),
            StructField("previousBalance", DoubleType(), False),
            StructField("isFraud", BooleanType(), False)
        ])
    
    def consume_and_create_df(self, batch_size=2000):
        messages = self.kafka.consume("transactions", "fraud-group", batch_size)
        if not messages:
            return None
            
        transactions = [json.loads(m["value"]) for m in messages]
        rows = [
            (t["transactionId"], t["customerId"], t["merchantId"],
             float(t["amount"]), t["currency"], t["channel"], t["location"],
             t["timestamp"], t["cardType"], t["isInternational"],
             t["merchantCategory"], float(t["previousBalance"]), t["isFraud"])
            for t in transactions
        ]
        return self.spark.createDataFrame(rows, self.schema)
    
    def enrich(self, df):
        """Enrichir avec features calculées"""
        return df \
            .withColumn("hour", (F.col("timestamp") / 3600000 % 24).cast("int")) \
            .withColumn("dayOfWeek", F.dayofweek(F.from_unixtime(F.col("timestamp") / 1000))) \
            .withColumn("isSuspiciousHour", F.col("hour").between(0, 5)) \
            .withColumn("isRiskyMerchant", 
                F.col("merchantCategory").isin("GAMBLING", "CRYPTO")) \
            .withColumn("amountCategory",
                F.when(F.col("amount") < 100, "LOW")
                .when(F.col("amount") < 500, "MEDIUM")
                .when(F.col("amount") < 2000, "HIGH")
                .otherwise("VERY_HIGH")
            )
    
    def apply_fraud_rules(self, df):
        """Appliquer règles de détection"""
        return df \
            .withColumn("fraudScore",
                F.when(F.col("amount") > 10000, F.lit(0.5))
                .when(F.col("amount") > 5000, F.lit(0.3))
                .otherwise(F.lit(0.0)) +
                F.when(F.col("isRiskyMerchant"), F.lit(0.3)).otherwise(F.lit(0.0)) +
                F.when(F.col("isSuspiciousHour"), F.lit(0.15)).otherwise(F.lit(0.0)) +
                F.when(F.col("isInternational") & (F.col("amount") > 1000), F.lit(0.2))
                .otherwise(F.lit(0.0))
            ) \
            .withColumn("riskLevel",
                F.when(F.col("fraudScore") >= 0.7, "CRITICAL")
                .when(F.col("fraudScore") >= 0.5, "HIGH")
                .when(F.col("fraudScore") >= 0.3, "MEDIUM")
                .otherwise("LOW")
            ) \
            .withColumn("isFraudPredicted", F.col("fraudScore") >= 0.5)
    
    def process_batch(self, batch_id):
        print(f"\n{'─'*50}")
        print(f"⚡ Batch {batch_id}")
        
        df = self.consume_and_create_df()
        if df is None:
            print("   Pas de données")
            return None
        
        count = df.count()
        print(f"   📥 {count} messages de Kafka")
        
        df = self.enrich(df)
        df = self.apply_fraud_rules(df)
        
        fraud_pred = df.filter(F.col("isFraudPredicted")).count()
        actual = df.filter(F.col("isFraud")).count()
        
        print(f"   🚨 Fraudes détectées: {fraud_pred}")
        print(f"   ✓ Vraies fraudes: {actual}")
        
        self.processed_count += count
        self.alerts_count += fraud_pred
        
        return df

processor = SparkStreamingProcessor(spark, kafka)

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# EXÉCUTER LE STREAMING
# ══════════════════════════════════════════════════════════════════════

print("="*60)
print("   SPARK STRUCTURED STREAMING")
print("="*60)

all_dfs = []
for batch_id in range(5):
    df = processor.process_batch(batch_id + 1)
    if df:
        all_dfs.append(df)

# Combiner
final_df = all_dfs[0]
for df in all_dfs[1:]:
    final_df = final_df.union(df)

print(f"\n{'='*60}")
print(f"✅ Total traité: {processor.processed_count}")
print(f"   Alertes: {processor.alerts_count}")

In [ ]:
# Afficher échantillon
print("\n📋 Échantillon:")
final_df.select(
    "transactionId", "amount", "merchantCategory",
    "fraudScore", "riskLevel", "isFraudPredicted", "isFraud"
).show(10, truncate=False)

---
# 💾 PARTIE 3: HDFS STORAGE (Parquet)

In [ ]:
print("="*60)
print("   HDFS - Sauvegarde Parquet partitionné")
print("="*60)

# Ajouter colonne date
final_df = final_df.withColumn(
    "transaction_date", 
    F.to_date(F.from_unixtime(F.col("timestamp") / 1000))
)

# Sauvegarder partitionné
final_df.write \
    .mode("overwrite") \
    .partitionBy("riskLevel", "transaction_date") \
    .parquet(HDFS_PROCESSED)

print(f"\n✅ Sauvegardé: {HDFS_PROCESSED}")

# Afficher structure
print("\n📁 Structure:")
for root, dirs, files in os.walk(HDFS_PROCESSED):
    level = root.replace(HDFS_PROCESSED, '').count(os.sep)
    indent = '  ' * level
    folder = os.path.basename(root)
    if folder and not folder.startswith('.'):
        print(f"{indent}📁 {folder}/")

In [ ]:
# Sauvegarder alertes + publier sur Kafka
alerts_df = final_df.filter(F.col("riskLevel").isin("HIGH", "CRITICAL"))
alerts_df.write.mode("overwrite").parquet(HDFS_ALERTS)

alert_count = alerts_df.count()
print(f"\n🚨 {alert_count} alertes sauvegardées")

# Publier alertes vers Kafka
for row in alerts_df.select("transactionId", "customerId", "amount", "fraudScore").collect():
    kafka.produce("fraud-alerts", row.transactionId, json.dumps(row.asDict()))

print(f"📤 {alert_count} alertes publiées sur 'fraud-alerts'")

---
# 🔍 PARTIE 4: IMPALA SQL (Requêtes Massives)

In [ ]:
# Charger depuis HDFS et créer table
df_hdfs = spark.read.parquet(HDFS_PROCESSED)
df_hdfs.createOrReplaceTempView("transactions")

print(f"✅ Table 'transactions' créée: {df_hdfs.count()} lignes")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# REQUÊTE 1: Métriques Dashboard
# ══════════════════════════════════════════════════════════════════════

print("📊 REQUÊTE 1: Métriques globales")
print("─"*50)

spark.sql("""
    SELECT
        COUNT(*) as total_tx,
        ROUND(SUM(amount), 2) as volume,
        ROUND(AVG(amount), 2) as avg_amount,
        COUNT(DISTINCT customerId) as customers,
        SUM(CASE WHEN isFraud THEN 1 ELSE 0 END) as real_frauds,
        SUM(CASE WHEN isFraudPredicted THEN 1 ELSE 0 END) as predicted_frauds
    FROM transactions
""").show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# REQUÊTE 2: Par catégorie marchand
# ══════════════════════════════════════════════════════════════════════

print("📊 REQUÊTE 2: Analyse par catégorie")
print("─"*50)

spark.sql("""
    SELECT
        merchantCategory,
        COUNT(*) as tx_count,
        ROUND(SUM(amount), 2) as volume,
        SUM(CASE WHEN isFraud THEN 1 ELSE 0 END) as frauds,
        ROUND(SUM(CASE WHEN isFraud THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as fraud_rate
    FROM transactions
    GROUP BY merchantCategory
    ORDER BY fraud_rate DESC
""").show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# REQUÊTE 3: Par niveau de risque
# ══════════════════════════════════════════════════════════════════════

print("📊 REQUÊTE 3: Distribution risque")
print("─"*50)

spark.sql("""
    SELECT
        riskLevel,
        COUNT(*) as count,
        ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM transactions), 2) as pct,
        ROUND(AVG(fraudScore), 4) as avg_score
    FROM transactions
    GROUP BY riskLevel
    ORDER BY avg_score DESC
""").show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# REQUÊTE 4: Top clients (CTE)
# ══════════════════════════════════════════════════════════════════════

print("📊 REQUÊTE 4: Top clients")
print("─"*50)

spark.sql("""
    WITH stats AS (
        SELECT
            customerId,
            COUNT(*) as tx_count,
            ROUND(SUM(amount), 2) as total,
            ROUND(AVG(fraudScore), 4) as avg_score
        FROM transactions
        GROUP BY customerId
    )
    SELECT * FROM stats ORDER BY total DESC LIMIT 10
""").show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# REQUÊTE 5: Matrice confusion
# ══════════════════════════════════════════════════════════════════════

print("📊 REQUÊTE 5: Matrice de confusion")
print("─"*50)

confusion = spark.sql("""
    SELECT
        SUM(CASE WHEN isFraudPredicted AND isFraud THEN 1 ELSE 0 END) as TP,
        SUM(CASE WHEN isFraudPredicted AND NOT isFraud THEN 1 ELSE 0 END) as FP,
        SUM(CASE WHEN NOT isFraudPredicted AND isFraud THEN 1 ELSE 0 END) as FN,
        SUM(CASE WHEN NOT isFraudPredicted AND NOT isFraud THEN 1 ELSE 0 END) as TN
    FROM transactions
""").collect()[0]

tp, fp, fn, tn = confusion.TP, confusion.FP, confusion.FN, confusion.TN
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f"  TP={tp} | FP={fp}")
print(f"  FN={fn} | TN={tn}")
print(f"\n  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1-Score:  {f1:.4f}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# REQUÊTE 6: Window Functions (vélocité)
# ══════════════════════════════════════════════════════════════════════

print("📊 REQUÊTE 6: Window Functions - Vélocité")
print("─"*50)

spark.sql("""
    SELECT
        customerId,
        transactionId,
        amount,
        ROW_NUMBER() OVER (PARTITION BY customerId ORDER BY timestamp) as tx_num,
        SUM(amount) OVER (PARTITION BY customerId ORDER BY timestamp) as cumul_amount,
        COUNT(*) OVER (PARTITION BY customerId) as total_tx
    FROM transactions
    WHERE customerId IN (
        SELECT customerId FROM transactions 
        GROUP BY customerId HAVING COUNT(*) >= 3
        LIMIT 3
    )
    ORDER BY customerId, tx_num
""").show(15)

---
# 🤖 PARTIE 5: MACHINE LEARNING (MLlib)

In [ ]:
print("="*60)
print("   SPARK MLlib - Random Forest")
print("="*60)

# Préparer données
ml_data = df_hdfs.select(
    "amount", "previousBalance", "hour", "fraudScore",
    "channel", "cardType", "merchantCategory",
    F.col("isFraud").cast("double").alias("label")
).na.fill(0)

train, test = ml_data.randomSplit([0.8, 0.2], seed=42)
print(f"\nTrain: {train.count()}, Test: {test.count()}")

In [ ]:
# Pipeline ML
indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}Idx", handleInvalid="keep")
    for c in ["channel", "cardType", "merchantCategory"]
]

assembler = VectorAssembler(
    inputCols=["amount", "previousBalance", "hour", "fraudScore",
               "channelIdx", "cardTypeIdx", "merchantCategoryIdx"],
    outputCol="features", handleInvalid="skip"
)

scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures")

rf = RandomForestClassifier(
    featuresCol="scaledFeatures", labelCol="label",
    numTrees=50, maxDepth=10, seed=42
)

pipeline = Pipeline(stages=indexers + [assembler, scaler, rf])

print("🚀 Entraînement...")
model = pipeline.fit(train)
print("✅ Modèle entraîné!")

In [ ]:
# Évaluation
predictions = model.transform(test)

evaluator = BinaryClassificationEvaluator(labelCol="label")
auc = evaluator.evaluate(predictions, {evaluator.metricName: "areaUnderROC"})

# Métriques
tp = predictions.filter((F.col("prediction") == 1) & (F.col("label") == 1)).count()
fp = predictions.filter((F.col("prediction") == 1) & (F.col("label") == 0)).count()
fn = predictions.filter((F.col("prediction") == 0) & (F.col("label") == 1)).count()
tn = predictions.filter((F.col("prediction") == 0) & (F.col("label") == 0)).count()

ml_precision = tp / (tp + fp) if (tp + fp) > 0 else 0
ml_recall = tp / (tp + fn) if (tp + fn) > 0 else 0
ml_f1 = 2 * ml_precision * ml_recall / (ml_precision + ml_recall) if (ml_precision + ml_recall) > 0 else 0

print(f"\n{'='*50}")
print(f"       RÉSULTATS ML")
print(f"{'='*50}")
print(f"AUC-ROC:    {auc:.4f}")
print(f"Precision:  {ml_precision:.4f}")
print(f"Recall:     {ml_recall:.4f}")
print(f"F1-Score:   {ml_f1:.4f}")
print(f"{'─'*50}")
print(f"TP={tp} | FP={fp}")
print(f"FN={fn} | TN={tn}")

In [ ]:
# Sauvegarder modèle
model.write().overwrite().save(f"{HDFS_MODELS}/fraud_rf")
print(f"\n✅ Modèle sauvegardé: {HDFS_MODELS}/fraud_rf")

---
# 📊 PARTIE 6: VISUALISATION

In [ ]:
pdf = df_hdfs.toPandas()
print(f"✅ Données Pandas: {len(pdf)} lignes")

In [ ]:
# Dashboard 1
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=("Distribution Montants", "Par Catégorie", 
                    "Score Fraude", "Niveau Risque"),
    specs=[[{"type": "histogram"}, {"type": "bar"}],
           [{"type": "histogram"}, {"type": "pie"}]]
)

fig.add_trace(go.Histogram(x=pdf['amount'], nbinsx=50, marker_color='#3498db'), row=1, col=1)

cat_counts = pdf['merchantCategory'].value_counts()
fig.add_trace(go.Bar(x=cat_counts.index, y=cat_counts.values, marker_color='#2ecc71'), row=1, col=2)

fig.add_trace(go.Histogram(x=pdf['fraudScore'], nbinsx=30, marker_color='#e74c3c'), row=2, col=1)

risk_counts = pdf['riskLevel'].value_counts()
colors = {'LOW': '#2ecc71', 'MEDIUM': '#f39c12', 'HIGH': '#e67e22', 'CRITICAL': '#e74c3c'}
fig.add_trace(go.Pie(labels=risk_counts.index, values=risk_counts.values,
                     marker_colors=[colors.get(r, '#95a5a6') for r in risk_counts.index]), row=2, col=2)

fig.update_layout(height=600, title_text="📊 Dashboard", showlegend=False)
fig.show()

In [ ]:
# Matrice confusion visuelle
confusion_matrix = [[tn, fp], [fn, tp]]

fig2 = go.Figure(data=go.Heatmap(
    z=confusion_matrix,
    x=['Prédit: Non-Fraude', 'Prédit: Fraude'],
    y=['Réel: Non-Fraude', 'Réel: Fraude'],
    text=[[f'TN={tn}', f'FP={fp}'], [f'FN={fn}', f'TP={tp}']],
    texttemplate="%{text}",
    colorscale='RdYlGn_r'
))

fig2.update_layout(title="🎯 Matrice Confusion", height=400)
fig2.show()

---
# 📅 PARTIE 7: AIRFLOW (DAG)

In [ ]:
class AirflowDAG:
    def __init__(self, dag_id):
        self.dag_id = dag_id
        self.tasks = []
        self.status = {}
        
    def add_task(self, task_id, func, depends=[]):
        self.tasks.append({"id": task_id, "func": func, "deps": depends})
        self.status[task_id] = "pending"
        
    def run(self):
        print(f"\n{'='*60}")
        print(f"   AIRFLOW DAG: {self.dag_id}")
        print(f"   Start: {datetime.now().strftime('%H:%M:%S')}")
        print(f"{'='*60}\n")
        
        for task in self.tasks:
            deps_ok = all(self.status.get(d) == "success" for d in task["deps"])
            if not deps_ok:
                print(f"⏭️  [{task['id']}] Skipped")
                continue
            
            print(f"▶️  [{task['id']}] Running...")
            try:
                result = task["func"]()
                self.status[task["id"]] = "success"
                print(f"✅ [{task['id']}] Done: {result}")
            except Exception as e:
                self.status[task["id"]] = "failed"
                print(f"❌ [{task['id']}] Error: {e}")
        
        print(f"\n{'='*60}")
        print(f"   DAG Complete")
        print(f"{'='*60}")

# Créer DAG
dag = AirflowDAG("fraud_detection_pipeline")

dag.add_task("check_kafka", lambda: f"{kafka.get_stats('transactions')['total_messages']} msgs")
dag.add_task("process_data", lambda: f"{processor.processed_count} traités", ["check_kafka"])
dag.add_task("save_hdfs", lambda: "Parquet sauvé", ["process_data"])
dag.add_task("refresh_tables", lambda: "Tables rafraîchies", ["save_hdfs"])
dag.add_task("send_alerts", lambda: f"{kafka.get_stats('fraud-alerts')['total_messages']} alertes", ["refresh_tables"])

dag.run()

---
# 🎉 RÉSUMÉ

In [ ]:
print("""
╔════════════════════════════════════════════════════════════════╗
║                 ARCHITECTURE DÉMONTRÉE                         ║
╠════════════════════════════════════════════════════════════════╣
║  1. KAFKA      → Topics, Producer, Consumer                   ║
║  2. SPARK RDD  → map, filter, reduceByKey                     ║
║  3. DATAFRAME  → withColumn, groupBy, Window Functions        ║
║  4. SPARK SQL  → Requêtes complexes, CTE, agrégations        ║
║  5. STREAMING  → Micro-batch processing                       ║
║  6. HDFS       → Parquet partitionné                          ║
║  7. IMPALA     → SQL distribué (via Spark SQL)                ║
║  8. MLlib      → Random Forest                                ║
║  9. AIRFLOW    → DAG orchestration                            ║
║ 10. DASHBOARD  → Visualisation Plotly                         ║
╚════════════════════════════════════════════════════════════════╝
""")

print(f"\n📊 STATISTIQUES FINALES:")
print(f"   • Transactions: {len(all_transactions)}")
print(f"   • Messages Kafka: {kafka.get_stats('transactions')['total_messages']}")
print(f"   • Alertes: {kafka.get_stats('fraud-alerts')['total_messages']}")
print(f"   • Precision ML: {ml_precision:.2%}")
print(f"   • Recall ML: {ml_recall:.2%}")

In [ ]:
spark.stop()
print("\n✅ Terminé!")